In [1]:
!pip install -q pandas numpy requests beautifulsoup4 tqdm lxml

In [2]:
import pandas as pd
import numpy as np
import requests
import re
import time

from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import datetime
from tqdm import tqdm

In [3]:
OUTPUT_FILE = "snack_dessert_scrape.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

KATALOG_URLS = [
    {
        "url": "https://menukuliner.net/katalog/roti-bakar",
        "raw_category": "roti bakar",
        "city": "unknown",
        "keyword": "roti bakar"
    },
    {
        "url": "https://menukuliner.net/katalog/croffle",
        "raw_category": "croffle",
        "city": "unknown",
        "keyword": "croffle"
    },
    {
        "url": "https://menukuliner.net/katalog/waffle",
        "raw_category": "waffle",
        "city": "unknown",
        "keyword": "waffle"
    },
    {
        "url": "https://menukuliner.net/katalog/pancake",
        "raw_category": "pancake",
        "city": "unknown",
        "keyword": "pancake"
    },
    {
        "url": "https://menukuliner.net/katalog/dessert-box",
        "raw_category": "dessert box",
        "city": "unknown",
        "keyword": "dessert box"
    },
    {
        "url": "https://menukuliner.net/katalog/cheesecake",
        "raw_category": "cheesecake",
        "city": "unknown",
        "keyword": "cheesecake"
    },
    {
        "url": "https://menukuliner.net/katalog/cake",
        "raw_category": "cake",
        "city": "unknown",
        "keyword": "cake"
    },
    {
        "url": "https://menukuliner.net/katalog/bakery",
        "raw_category": "bakery",
        "city": "unknown",
        "keyword": "bakery"
    },
    {
        "url": "https://menukuliner.net/katalog/roti",
        "raw_category": "bakery / bread",
        "city": "unknown",
        "keyword": "roti"
    },
    {
        "url": "https://menukuliner.net/katalog/kentang-goreng",
        "raw_category": "snack",
        "city": "unknown",
        "keyword": "kentang goreng"
    },
    {
        "url": "https://menukuliner.net/katalog/burger",
        "raw_category": "burger",
        "city": "unknown",
        "keyword": "burger"
    },
    {
        "url": "https://menukuliner.net/katalog/es-krim",
        "raw_category": "ice cream",
        "city": "unknown",
        "keyword": "es krim"
    },
    {
        "url": "https://menukuliner.net/katalog/coklat",
        "raw_category": "chocolate",
        "city": "unknown",
        "keyword": "coklat"
    },
]

MAX_LINKS = 100
REQUEST_DELAY = 1

In [4]:
def clean_text(text):
    if text is None:
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def clean_price(text):
    if text is None or pd.isna(text):
        return np.nan

    text = str(text)
    text = re.sub(r"[^0-9]", "", text)

    if text == "":
        return np.nan

    return int(text)


def is_price(text):
    text = clean_text(text)
    return bool(re.search(r"Rp\s*[0-9][0-9\.\,]*", text))


def is_valid_menu_name(text):
    text = clean_text(text)
    lower = text.lower()

    if len(text) < 3:
        return False

    if len(text) > 120:
        return False

    if lower.startswith("rp"):
        return False

    if re.fullmatch(r"[0-9\.\,\s]+", text):
        return False

    noise_words = [
        "nama menu harga",
        "nama menu",
        "harga",
        "harga menu",
        "daftar harga",
        "delivery",
        "gofood",
        "gojek",
        "grabfood",
        "shopeefood",
        "menukuliner",
        "restaurant",
        "restoran",
        "promo",
        "diskon",
        "review",
        "rating",
        "alamat",
        "jam buka",
        "telepon",
        "cukup merogoh",
        "dibanderol",
        "disajikan",
        "berkisar",
        "pilihan menu",
        "siapkan uang",
        "tidak mahal",
        "untuk menyantap",
        "harga yang dibanderol",
        "anda bisa",
        "di bawah ini",
        "berikut ini",
        "terbaru",
        "halaman",
        "lihat",
        "baca juga",
    ]

    if any(word in lower for word in noise_words):
        return False

    return True


def infer_raw_category(text):
    text = str(text).lower()

    if any(k in text for k in ["roti bakar", "ropang", "toast", "roti panggang"]):
        return "roti bakar / toast"

    if any(k in text for k in ["croffle"]):
        return "croffle"

    if any(k in text for k in ["waffle"]):
        return "waffle"

    if any(k in text for k in ["pancake"]):
        return "pancake"

    if any(k in text for k in ["dessert box", "dessert"]):
        return "dessert"

    if any(k in text for k in ["cheesecake", "cheese cake"]):
        return "cheesecake"

    if any(k in text for k in ["cake", "brownies", "bolu", "kue"]):
        return "cake / pastry"

    if any(k in text for k in ["bakery", "roti", "croissant", "donut", "donat", "pastry"]):
        return "bakery"

    if any(k in text for k in ["kentang", "fries", "french fries", "sosis", "otak-otak"]):
        return "snack"

    if any(k in text for k in ["burger", "hotdog", "sandwich"]):
        return "burger / sandwich"

    if any(k in text for k in ["ice cream", "es krim", "gelato"]):
        return "ice cream"

    if any(k in text for k in ["coklat", "chocolate", "choco"]):
        return "chocolate snack"

    return "snack / dessert"


def infer_city_from_text(text):
    text = str(text).lower()

    cities = [
        "jakarta", "semarang", "bandung", "yogyakarta", "surabaya",
        "medan", "tangerang", "bekasi", "bogor", "depok",
        "malang", "solo", "denpasar", "balikpapan", "makassar"
    ]

    for city in cities:
        if city in text:
            return city.title()

    return "Unknown"

In [5]:
def collect_menu_links():
    collected = []

    for katalog in tqdm(KATALOG_URLS, desc="Collecting menu links"):
        url = katalog["url"]

        try:
            response = requests.get(url, headers=HEADERS, timeout=30)

            if response.status_code != 200:
                print(f"Skip {url} | status {response.status_code}")
                continue

            soup = BeautifulSoup(response.text, "lxml")

            for a in soup.find_all("a", href=True):
                href = a["href"]

                if "/menu/" not in href:
                    continue

                full_url = urljoin("https://menukuliner.net", href)

                if "menukuliner.net/menu/" not in full_url:
                    continue

                collected.append({
                    "source_platform": "MenuKuliner",
                    "source_url": full_url,
                    "raw_category_from_katalog": katalog["raw_category"],
                    "keyword": katalog["keyword"],
                    "city_from_katalog": katalog["city"]
                })

        except Exception as e:
            print(f"Failed katalog: {url} | {e}")

        time.sleep(REQUEST_DELAY)

    df_links = pd.DataFrame(collected)

    if df_links.empty:
        return df_links

    df_links = df_links.drop_duplicates(subset=["source_url"]).reset_index(drop=True)
    df_links = df_links.head(MAX_LINKS)

    return df_links


df_links = collect_menu_links()

print(df_links.shape)
display(df_links.head(20))

Skip https://menukuliner.net/katalog/croffle | status 404


Skip https://menukuliner.net/katalog/waffle | status 404


Skip https://menukuliner.net/katalog/pancake | status 404


Skip https://menukuliner.net/katalog/cheesecake | status 404


Skip https://menukuliner.net/katalog/cake | status 404


...

Skip https://menukuliner.net/katalog/kentang-goreng | status 404


Skip https://menukuliner.net/katalog/burger | status 404


Skip https://menukuliner.net/katalog/es-krim | status 404


Skip https://menukuliner.net/katalog/coklat | status 404
(80, 5)


,source_platform,source_url,raw_category_from_katalog,keyword,city_from_katalog
0,MenuKuliner,https://menukuliner.net/menu/299721/dapur-lari...,roti bakar,roti bakar,unknown
1,MenuKuliner,https://menukuliner.net/menu/346461/jajandulud...,roti bakar,roti bakar,unknown
2,MenuKuliner,https://menukuliner.net/menu/384484/kopi-timur,roti bakar,roti bakar,unknown
3,MenuKuliner,https://menukuliner.net/menu/551477/soto-tangk...,roti bakar,roti bakar,unknown
4,MenuKuliner,https://menukuliner.net/menu/584489/warung-bu-pat,roti bakar,roti bakar,unknown
5,MenuKuliner,https://menukuliner.net/menu/601532/warung-umm...,roti bakar,roti bakar,unknown
6,MenuKuliner,https://menukuliner.net/menu/720209/ayam-gepre...,roti bakar,roti bakar,unknown
7,MenuKuliner,https://menukuliner.net/menu/746003/warung-cem...,roti bakar,roti bakar,unknown
8,MenuKuliner,https://menukuliner.net/menu/773881/jajanan-an...,roti bakar,roti bakar,unknown
9,MenuKuliner,https://menukuliner.net/menu/870911/roti-bakar...,roti bakar,roti bakar,unknown


In [6]:
def get_restaurant_name(soup):
    h1 = soup.find("h1")

    if h1:
        title = clean_text(h1.get_text())
    else:
        title_tag = soup.find("title")
        title = clean_text(title_tag.get_text()) if title_tag else "Unknown Restaurant"

    title = title.replace("Daftar Harga Menu Delivery", "")
    title = title.replace("Daftar Harga Menu", "")
    title = title.replace("Terbaru", "")
    title = re.sub(r"\s+", " ", title)
    title = title.strip(" ,-")

    if title == "":
        return "Unknown Restaurant"

    return title


def extract_menu_from_table(soup):
    items = []

    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            cells = [clean_text(td.get_text()) for td in tr.find_all(["td", "th"])]

            if len(cells) < 2:
                continue

            name_candidate = cells[0]
            price_candidate = cells[-1]

            if not is_price(price_candidate):
                continue

            if not is_valid_menu_name(name_candidate):
                continue

            items.append({
                "section": None,
                "menu_name": name_candidate,
                "price": clean_price(price_candidate),
                "extract_method": "html_table"
            })

    return items


def extract_menu_from_text(soup):
    text = soup.get_text("\n")
    lines = [clean_text(line) for line in text.splitlines()]
    lines = [line for line in lines if line]

    items = []
    current_section = None

    for idx, line in enumerate(lines):
        if line.lower().startswith("harga menu"):
            current_section = clean_text(line.replace("Harga Menu", ""))
            continue

        if not is_price(line):
            continue

        price = clean_price(line)

        if pd.isna(price):
            continue

        candidates = []

        if idx - 1 >= 0:
            candidates.append(lines[idx - 1])

        if idx - 2 >= 0:
            candidates.append(lines[idx - 2])

        menu_name = None

        for candidate in candidates:
            if is_valid_menu_name(candidate):
                menu_name = candidate
                break

        if menu_name is None:
            continue

        items.append({
            "section": current_section,
            "menu_name": menu_name,
            "price": price,
            "extract_method": "text_pattern"
        })

    return items


def deduplicate_menu_items(items):
    unique = []
    seen = set()

    for item in items:
        if pd.isna(item["price"]):
            continue

        key = (
            str(item.get("section")).lower(),
            item["menu_name"].lower(),
            int(item["price"])
        )

        if key not in seen:
            seen.add(key)
            unique.append(item)

    return unique


def scrape_menu_page(target):
    rows = []
    url = target["source_url"]

    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "lxml")

        restaurant_name = get_restaurant_name(soup)

        menu_items = []
        menu_items.extend(extract_menu_from_table(soup))
        menu_items.extend(extract_menu_from_text(soup))
        menu_items = deduplicate_menu_items(menu_items)

        city = infer_city_from_text(restaurant_name + " " + url)

        if city == "Unknown" and target["city_from_katalog"] != "unknown":
            city = str(target["city_from_katalog"]).title()

        for item in menu_items:
            raw_category = infer_raw_category(
                str(target["raw_category_from_katalog"]) + " " +
                str(target["keyword"]) + " " +
                str(item["section"]) + " " +
                str(item["menu_name"])
            )

            rows.append({
                "restaurant_name": restaurant_name,
                "city": city,
                "raw_category": raw_category,
                "section": item["section"],
                "menu_name": item["menu_name"],
                "price": item["price"],
                "source_platform": target["source_platform"],
                "source_url": url,
                "search_keyword": target["keyword"],
                "extract_method": item["extract_method"],
                "scraped_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            })

    except Exception as e:
        print(f"Failed: {url} | {e}")

    return rows

In [7]:
all_rows = []

for _, target in tqdm(df_links.iterrows(), total=len(df_links), desc="Scraping menu pages"):
    rows = scrape_menu_page(target)
    all_rows.extend(rows)

    if all_rows:
        pd.DataFrame(all_rows).to_csv("checkpoint_snack_dessert_scrape.csv", index=False)

    time.sleep(REQUEST_DELAY)

df_snack_dessert = pd.DataFrame(all_rows)

if df_snack_dessert.empty:
    print("Tidak ada data berhasil discrape.")
else:
    df_snack_dessert = df_snack_dessert.drop_duplicates(
        subset=["restaurant_name", "menu_name", "price", "source_url"],
        keep="first"
    ).reset_index(drop=True)

    df_snack_dessert["price"] = pd.to_numeric(df_snack_dessert["price"], errors="coerce")

    df_snack_dessert = df_snack_dessert[
        (df_snack_dessert["price"].isna()) |
        ((df_snack_dessert["price"] >= 3000) & (df_snack_dessert["price"] <= 500000))
    ].copy()

    df_snack_dessert = df_snack_dessert.reset_index(drop=True)

    df_snack_dessert.to_csv(OUTPUT_FILE, index=False)

    print("=" * 80)
    print("SCRAPING SNACK / DESSERT SELESAI")
    print("=" * 80)
    print(f"Output file        : {OUTPUT_FILE}")
    print(f"Total rows         : {len(df_snack_dessert)}")
    print(f"Unique restaurants : {df_snack_dessert['restaurant_name'].nunique()}")
    print(f"Unique cities      : {df_snack_dessert['city'].nunique()}")
    print(f"Source platform    : {df_snack_dessert['source_platform'].unique().tolist()}")
    print(f"Min price          : {df_snack_dessert['price'].min()}")
    print(f"Median price       : {df_snack_dessert['price'].median()}")
    print(f"Max price          : {df_snack_dessert['price'].max()}")

    display(df_snack_dessert.head(30))

Scraping menu pages: 100%|██████████| 80/80 [02:10<00:00,  1.63s/it]

SCRAPING SNACK / DESSERT SELESAI
Output file        : snack_dessert_scrape.csv
Total rows         : 5163
Unique restaurants : 80
Unique cities      : 11
Source platform    : ['MenuKuliner']
Min price          : 3000
Median price       : 22500.0
Max price          : 500000


,restaurant_name,city,raw_category,section,menu_name,price,source_platform,source_url,search_keyword,extract_method,scraped_at
0,"Dapur Larisz Momih, Pondok Jaya, Jakarta 2026",Jakarta,roti bakar / toast,None,Nasi Kebuli Karee Ayam Nasi Kebuli Dengan Tamb...,30000,MenuKuliner,https://menukuliner.net/menu/299721/dapur-lari...,roti bakar,html_table,2026-04-28 03:07:28
1,"Dapur Larisz Momih, Pondok Jaya, Jakarta 2026",Jakarta,roti bakar / toast,None,Roti Maryam Topping Coklat Meses Roti Maryam D...,15000,MenuKuliner,https://menukuliner.net/menu/299721/dapur-lari...,roti bakar,html_table,2026-04-28 03:07:28
2,"Dapur Larisz Momih, Pondok Jaya, Jakarta 2026",Jakarta,roti bakar / toast,None,Maryam Gunting Choco Glaze Maryam Gunting With...,17000,MenuKuliner,https://menukuliner.net/menu/299721/dapur-lari...,roti bakar,html_table,2026-04-28 03:07:28
3,"Dapur Larisz Momih, Pondok Jaya, Jakarta 2026",Jakarta,roti bakar / toast,None,Risoles Isi Pasta Kentang Risol Yang Isinya Pa...,5000,MenuKuliner,https://menukuliner.net/menu/299721/dapur-lari...,roti bakar,html_table,2026-04-28 03:07:28
4,"Dapur Larisz Momih, Pondok Jaya, Jakarta 2026",Jakarta,roti bakar / toast,None,Smooth Flurry Oreo Ukuran Medium Ice Cream Lem...,20000,MenuKuliner,https://menukuliner.net/menu/299721/dapur-lari...,roti bakar,html_table,2026-04-28 03:07:28
5,"Dapur Larisz Momih, Pondok Jaya, Jakarta 2026",Jakarta,roti bakar / toast,None,Smooth Flurry Vanilla Oreo Small Cup Ice Cream...,15000,MenuKuliner,https://menukuliner.net/menu/299721/dapur-lari...,roti bakar,html_table,2026-04-28 03:07:28
6,"Dapur Larisz Momih, Pondok Jaya, Jakarta 2026",Jakarta,roti bakar / toast,None,Roti Maryam Kuah Karee Daging Sapi Roti Maryam...,35000,MenuKuliner,https://menukuliner.net/menu/299721/dapur-lari...,roti bakar,html_table,2026-04-28 03:07:28
7,"Dapur Larisz Momih, Pondok Jaya, Jakarta 2026",Jakarta,roti bakar / toast,None,Nasi Kebuli Daging Sapi,30000,MenuKuliner,https://menukuliner.net/menu/299721/dapur-lari...,roti bakar,html_table,2026-04-28 03:07:28
8,"Dapur Larisz Momih, Pondok Jaya, Jakarta 2026",Jakarta,roti bakar / toast,None,Maryam Gunting Cheezy Milk Roti Maryam Dengan ...,16000,MenuKuliner,https://menukuliner.net/menu/299721/dapur-lari...,roti bakar,html_table,2026-04-28 03:07:28
9,"Dapur Larisz Momih, Pondok Jaya, Jakarta 2026",Jakarta,roti bakar / toast,None,Roti Maryam Kuah Karee Ayam Roti Maryam Yang G...,26000,MenuKuliner,https://menukuliner.net/menu/299721/dapur-lari...,roti bakar,html_table,2026-04-28 03:07:28


In [8]:
print("Shape:", df_snack_dessert.shape)

display(df_snack_dessert["raw_category"].value_counts())
display(df_snack_dessert["restaurant_name"].value_counts().head(20))
display(df_snack_dessert.sample(min(20, len(df_snack_dessert)), random_state=42))

Shape: (5163, 11)


,count
raw_category,
roti bakar / toast,2816
dessert,2331
waffle,15
croffle,1


,count
restaurant_name,
"Rifaza Kitchen, Fresh Market Emerald Bintaro, Jakarta 2026",226
"Toko Kopi Seduh, Pesanggrahan, Jakarta 2026",207
"Anita Family Bakery, Sampang, Madura 2026",182
"Roti Cari Rasa Kosambi, Cicalengka, Bandung 2026",146
"Mahkota Frozen Food, Meruya, Jakarta 2026",146
"De Patata, Pasar Ambacang, Padang 2026",141
"Masalalu Café, Cimahi, Bandung 2026",133
"Roti Bakar Mbak Retno, Semarang Utara, Semarang 2026",124
"Bittersweet By Najla, Surabaya, Surabaya 2026",120


,restaurant_name,city,raw_category,section,menu_name,price,source_platform,source_url,search_keyword,extract_method,scraped_at
1726,"Roti Cari Rasa Kosambi, Cicalengka, Bandung 2026",Bandung,roti bakar / toast,None,Roti Bakar Kacang Selai Kacang+susu,23750,MenuKuliner,https://menukuliner.net/menu/120205/roti-cari-...,roti bakar,html_table,2026-04-28 03:08:14
1666,"Roti Bakar 25 Puff Pastry, Bandung 2026",Bandung,roti bakar / toast,None,Roti Bakar Tiramisu,26000,MenuKuliner,https://menukuliner.net/menu/119376/roti-bakar...,roti bakar,html_table,2026-04-28 03:08:11
4230,"Cafe Barcelona, S Parman, Batam 2026",Unknown,dessert,None,Kerak Telor Makanan,13000,MenuKuliner,https://menukuliner.net/menu/155379/cafe-barce...,dessert box,html_table,2026-04-28 03:09:19
1181,"INCORNER COFFEE, Cicaheum, Bandung 2026",Bandung,roti bakar / toast,None,"Milkshake Hazelnut Real Milk, Real Ice Cream, ...",21250,MenuKuliner,https://menukuliner.net/menu/90478/incorner-co...,roti bakar,html_table,2026-04-28 03:08:01
3129,"Mahkota Frozen Food, Meruya, Jakarta 2026",Jakarta,dessert,None,Champ ABC Chicken Nuggget 250g,24000,MenuKuliner,https://menukuliner.net/menu/395860/mahkota-fr...,dessert box,html_table,2026-04-28 03:08:46
4829,"Melati Bolu, Pulo Gadung, Jakarta 2026",Jakarta,dessert,None,Bolu Kacang,43000,MenuKuliner,https://menukuliner.net/menu/410360/melati-bol...,dessert box,html_table,2026-04-28 03:09:32
290,"Warung Cemal Cemil H & H, Elang, Medan 2026",Medan,roti bakar / toast,None,Dimsum Ayam Udang Isi 20,75000,MenuKuliner,https://menukuliner.net/menu/746003/warung-cem...,roti bakar,html_table,2026-04-28 03:07:39
1220,"INCORNER COFFEE, Cicaheum, Bandung 2026",Bandung,roti bakar / toast,Aneka Maincourse,Nasi + Beef Sliced Bumbu Balado,38500,MenuKuliner,https://menukuliner.net/menu/90478/incorner-co...,roti bakar,text_pattern,2026-04-28 03:08:01
3048,"Bittersweet By Najla, Tanjung Duren (Delivery ...",Jakarta,dessert,None,"Turkish Cake coklat moist, mousses dan siraman...",85000,MenuKuliner,https://menukuliner.net/menu/263374/bitterswee...,dessert box,html_table,2026-04-28 03:08:43
4540,"Vita's Kitchen, Ciledug, Jakarta 2026",Jakarta,dessert,Dessert Box,"Brownies, Lotus Biscoff Mousse, Brownies, Whip...",55000,MenuKuliner,https://menukuliner.net/menu/572479/vitas-kitc...,dessert box,text_pattern,2026-04-28 03:09:25


In [9]:
print("Shape:", df_snack_dessert.shape)

display(df_snack_dessert["raw_category"].value_counts())
display(df_snack_dessert["restaurant_name"].value_counts().head(20))
display(df_snack_dessert.sample(min(20, len(df_snack_dessert)), random_state=42))

Shape: (5163, 11)


,count
raw_category,
roti bakar / toast,2816
dessert,2331
waffle,15
croffle,1


,count
restaurant_name,
"Rifaza Kitchen, Fresh Market Emerald Bintaro, Jakarta 2026",226
"Toko Kopi Seduh, Pesanggrahan, Jakarta 2026",207
"Anita Family Bakery, Sampang, Madura 2026",182
"Roti Cari Rasa Kosambi, Cicalengka, Bandung 2026",146
"Mahkota Frozen Food, Meruya, Jakarta 2026",146
"De Patata, Pasar Ambacang, Padang 2026",141
"Masalalu Café, Cimahi, Bandung 2026",133
"Roti Bakar Mbak Retno, Semarang Utara, Semarang 2026",124
"Bittersweet By Najla, Surabaya, Surabaya 2026",120


,restaurant_name,city,raw_category,section,menu_name,price,source_platform,source_url,search_keyword,extract_method,scraped_at
1726,"Roti Cari Rasa Kosambi, Cicalengka, Bandung 2026",Bandung,roti bakar / toast,None,Roti Bakar Kacang Selai Kacang+susu,23750,MenuKuliner,https://menukuliner.net/menu/120205/roti-cari-...,roti bakar,html_table,2026-04-28 03:08:14
1666,"Roti Bakar 25 Puff Pastry, Bandung 2026",Bandung,roti bakar / toast,None,Roti Bakar Tiramisu,26000,MenuKuliner,https://menukuliner.net/menu/119376/roti-bakar...,roti bakar,html_table,2026-04-28 03:08:11
4230,"Cafe Barcelona, S Parman, Batam 2026",Unknown,dessert,None,Kerak Telor Makanan,13000,MenuKuliner,https://menukuliner.net/menu/155379/cafe-barce...,dessert box,html_table,2026-04-28 03:09:19
1181,"INCORNER COFFEE, Cicaheum, Bandung 2026",Bandung,roti bakar / toast,None,"Milkshake Hazelnut Real Milk, Real Ice Cream, ...",21250,MenuKuliner,https://menukuliner.net/menu/90478/incorner-co...,roti bakar,html_table,2026-04-28 03:08:01
3129,"Mahkota Frozen Food, Meruya, Jakarta 2026",Jakarta,dessert,None,Champ ABC Chicken Nuggget 250g,24000,MenuKuliner,https://menukuliner.net/menu/395860/mahkota-fr...,dessert box,html_table,2026-04-28 03:08:46
4829,"Melati Bolu, Pulo Gadung, Jakarta 2026",Jakarta,dessert,None,Bolu Kacang,43000,MenuKuliner,https://menukuliner.net/menu/410360/melati-bol...,dessert box,html_table,2026-04-28 03:09:32
290,"Warung Cemal Cemil H & H, Elang, Medan 2026",Medan,roti bakar / toast,None,Dimsum Ayam Udang Isi 20,75000,MenuKuliner,https://menukuliner.net/menu/746003/warung-cem...,roti bakar,html_table,2026-04-28 03:07:39
1220,"INCORNER COFFEE, Cicaheum, Bandung 2026",Bandung,roti bakar / toast,Aneka Maincourse,Nasi + Beef Sliced Bumbu Balado,38500,MenuKuliner,https://menukuliner.net/menu/90478/incorner-co...,roti bakar,text_pattern,2026-04-28 03:08:01
3048,"Bittersweet By Najla, Tanjung Duren (Delivery ...",Jakarta,dessert,None,"Turkish Cake coklat moist, mousses dan siraman...",85000,MenuKuliner,https://menukuliner.net/menu/263374/bitterswee...,dessert box,html_table,2026-04-28 03:08:43
4540,"Vita's Kitchen, Ciledug, Jakarta 2026",Jakarta,dessert,Dessert Box,"Brownies, Lotus Biscoff Mousse, Brownies, Whip...",55000,MenuKuliner,https://menukuliner.net/menu/572479/vitas-kitc...,dessert box,text_pattern,2026-04-28 03:09:25


In [10]:
from google.colab import files

files.download("snack_dessert_scrape.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>